In [ ]:
shared_root <- Sys.getenv("REGA_SHARED_ROOT", unset = normalizePath(".."))
separate_root <- Sys.getenv("REGA_SEPARATE_ROOT", unset = "/project/zduren/durenlab/naqing_palmetto/REGA_separate")
library(tidyverse)

In [ ]:
LDSC_FILE <- file.path(shared_root, "part4_ldsc/heritability_results/AD_Bellenguez_single_module_extend2kb_LDSC_results.tsv")
OUT_DIR <- file.path(shared_root, "part4_ldsc")

In [ ]:
shared_root

In [ ]:
OUT_DIR

In [ ]:
# ============================================================
# 2. Selected cross-cell-type modules
# ============================================================

MODULE_ORDER <- trimws(readLines(file.path(shared_root, "part2_run_REGA/08pathology_tests/selected_module.txt"), warn = FALSE))
MODULE_ORDER <- MODULE_ORDER[nzchar(MODULE_ORDER)]


CELLTYPE_ORDER <- c(
  "Exc",
  "Inh",
  "Ast",
  "Oli",
  "OPC",
  "Mic_Immune"
)

# Check the current disease-related selection.
selected_module_file <- file.path(
  shared_root, "part2_run_REGA/08pathology_tests/selected_module.txt"
)
disease_modules <- trimws(readLines(selected_module_file))
disease_modules <- disease_modules[nzchar(disease_modules)]
stopifnot(length(MODULE_ORDER) > 0L, identical(MODULE_ORDER, disease_modules))

In [ ]:
# ============================================================
# 3. Region colors
# ============================================================

region_colors <- c(
  AG  = "#8DA0CB",
  EC  = "#FC8D62",
  HC  = "#66C2A5",
  MTC = "#E78AC3",
  PFC = "#B3B3B3",
  TH  = "#A6D854"
)


# ============================================================
# 4. Read LDSC results
# ============================================================

ldsc <- read.delim(
  LDSC_FILE,
  header = TRUE,
  sep = "\t",
  check.names = FALSE,
  stringsAsFactors = FALSE
)



In [ ]:
# ============================================================
# 5. Parse cell type / region / region_module
# ============================================================
#
# Examples:
#
# Mic_Immune_PFC_M19
# Exc_EC_M14
# Oli_MTC_M6
#
# celltype_region:
# Mic_Immune_PFC
# Exc_EC
#
# module:
# M19
# ============================================================

plot_df <- ldsc %>%

  mutate(

    # region = final component of celltype_region
    region = str_extract(
      celltype_region,
      "(AG|EC|HC|MTC|PFC|TH)$"
    ),

    # remove "_REGION" from celltype_region
    celltype = str_remove(
      celltype_region,
      paste0("_", region, "$")
    ),

    region_module = paste(
      region,
      module,
      sep = "_"
    )
  ) %>%

  # Keep only the current selected modules
  filter(
    region_module %in% MODULE_ORDER,
    celltype %in% CELLTYPE_ORDER
  ) %>%

  mutate(

    Enrichment = as.numeric(Enrichment),
    Enrichment_p = as.numeric(Enrichment_p),

    # --------------------------------------------------------
    # Bubble color
    # --------------------------------------------------------

    neglog10_p = -log10(
      pmax(
        Enrichment_p,
        .Machine$double.xmin
      )
    ),

    # --------------------------------------------------------
    # Bubble size
    #
    # Negative enrichment estimates cannot have negative
    # bubble sizes. Treat non-positive enrichment as zero.
    # --------------------------------------------------------

    enrichment_for_size = pmax(
      Enrichment,
      0
    ),

    region_module = factor(
      region_module,
      levels = MODULE_ORDER
    ),

    celltype = factor(
      celltype,
      levels = rev(CELLTYPE_ORDER)
    )
  )

# The collector already applies BH over the selected cell-type/module tests.
# Verify the table uses exactly the selected testing family.
stopifnot(
  nrow(plot_df) == length(MODULE_ORDER) * length(CELLTYPE_ORDER),
  all(is.finite(plot_df$Enrichment_p)),
  all(plot_df$Enrichment_p >= 0 & plot_df$Enrichment_p <= 1)
)
stopifnot(isTRUE(all.equal(
  as.numeric(plot_df$FDR_BH_Enrichment_p),
  as.numeric(p.adjust(plot_df$Enrichment_p, method = "BH")),
  tolerance = 1e-10
)))

# Draw dots only for contexts with nominal LDSC enrichment P < 0.05.
# Keep plot_df complete for the selected-context audit and exported table.
plot_sig_df <- plot_df %>%
  filter(
    is.finite(Enrichment_p),
    Enrichment_p >= 0,
    Enrichment_p < 0.05
  )


In [ ]:
# ============================================================
# 6. Validate selected data
# ============================================================

cat(
  "Number of rows:",
  nrow(plot_df),
  "\n"
)

cat(
  "Expected:",
  length(MODULE_ORDER) * length(CELLTYPE_ORDER),
  "\n\n"
)

cat("Nominal P < 0.05 dots:", nrow(plot_sig_df), "\n\n")
cat(
  "BH FDR < 0.05 among the selected tests:",
  sum(plot_df$FDR_BH_Enrichment_p < 0.05),
  "\n\n"
)


# Number of cell types per module
print(
  plot_df %>%
    count(region_module)
)


# Check missing combinations
missing_combinations <- tidyr::crossing(

  region_module = MODULE_ORDER,
  celltype = CELLTYPE_ORDER

) %>%

  anti_join(
    plot_df %>%
      transmute(
        region_module = as.character(region_module),
        celltype = as.character(celltype)
      ),
    by = c(
      "region_module",
      "celltype"
    )
  )


if (nrow(missing_combinations) > 0) {

  warning(
    "Missing celltype × module combinations:"
  )

  print(missing_combinations)

} else {

  message(
    "All selected celltype × module combinations are present."
  )
}

In [ ]:
# ============================================================
# 7. Show negative enrichment estimates
# ============================================================

negative_enrichment <- plot_df %>%
  filter(
    Enrichment <= 0
  ) %>%
  select(
    region_module,
    celltype,
    Enrichment,
    Enrichment_p,
    neglog10_p
  )


if (nrow(negative_enrichment) > 0) {

  message(
    "\nNon-positive enrichment estimates ",
    "(bubble size set to 0):"
  )

  print(negative_enrichment)
}

In [ ]:
# ============================================================
# 8. Save plotting table
# ============================================================

selected_plot_table <- plot_df %>%
  arrange(region_module, celltype)

# Save the same selected-family BH result beside the figure and the source table.
for (output_file in c(
  file.path(OUT_DIR, "selected_cross_celltype_modules_LDSC_bubbleplot_data.tsv"),
  file.path(dirname(LDSC_FILE), "selected_cross_celltype_modules_LDSC_bubbleplot_data.tsv")
)) {
  write.table(
    selected_plot_table,
    output_file,
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )
}


In [ ]:
# ============================================================
# 9. Region annotation for top strip
# ============================================================

region_annotation <- tibble(

  region_module = factor(
    MODULE_ORDER,
    levels = MODULE_ORDER
  ),

  region = str_extract(
    MODULE_ORDER,
    "^[^_]+"
  )
)


In [ ]:
# ============================================================
# 10. Main LDSC bubble plot
# ============================================================

p <- ggplot(
  plot_sig_df,
  aes(
    x = region_module,
    y = celltype
  )
) +

  # ----------------------------------------------------------
  # Bubble:
  # size  = enrichment
  # color = -log10(P)
  # ----------------------------------------------------------

  geom_point(
    aes(
      size = enrichment_for_size,
      color = neglog10_p
    ),
    alpha = 0.9
  ) +

  # Preserve every selected module and cell type when no dot passes.
  scale_x_discrete(limits = MODULE_ORDER, drop = FALSE) +
  scale_y_discrete(limits = rev(CELLTYPE_ORDER), drop = FALSE) +

  # ----------------------------------------------------------
  # Size scale
  #
  # scale_size_area ensures zero enrichment -> zero area
  # ----------------------------------------------------------

  scale_size_area(
    name = "Enrichment",
    max_size = 16,
    breaks = c(
      2,
      5,
      10,
      15
    )
  ) +

  # ----------------------------------------------------------
  # Color scale
  # ----------------------------------------------------------

  scale_color_gradientn(
    name = expression(-log[10](italic(P))),
    colours = c(
      "#D9D9D9",
      "#9ECAE1",
      "#4292C6",
      "#2171B5",
      "#084594"
    )
  ) +

  # ----------------------------------------------------------
  # Labels
  # ----------------------------------------------------------

  labs(
    x = NULL,
    y = NULL
  ) +

  # ----------------------------------------------------------
  # Theme
  # ----------------------------------------------------------

  theme_classic(
    base_size = 18
  ) +

  theme(

    axis.text.x = element_text(
      angle = 90,
      hjust = 1,
      vjust = 1,
      size = 15,
      color = "black"
    ),

    axis.text.y = element_text(
      size = 16,
      color = "black"
    ),

    axis.line = element_blank(),

    axis.ticks = element_blank(),

    legend.position = "right",

    legend.title = element_text(
      size = 15,
      face = "bold"
    ),

    legend.text = element_text(
      size = 13
    ),

    panel.grid.major = element_line(
      color = "grey90",
      linewidth = 0.4
    ),

    panel.grid.minor = element_blank(),

    plot.margin = margin(
      15,
      20,
      10,
      10
    )
  )


p

In [ ]:
# ============================================================
# 11. Region color strip
# ============================================================

library(patchwork)


region_strip <- ggplot(
  region_annotation,
  aes(
    x = region_module,
    y = 1,
    fill = region
  )
) +

  geom_tile(
    width = 0.95,
    height = 0.55
  ) +

  scale_fill_manual(
    values = region_colors,
    guide = "none"
  ) +

  scale_x_discrete(
    limits = MODULE_ORDER,
    drop = FALSE
  ) +

  coord_cartesian(
    ylim = c(0.7, 1.3),
    expand = FALSE
  ) +

  theme_void() +

  theme(
    plot.margin = margin(
      0,
      0,
      -5,
      0
    )
  )


# ============================================================
# 12. Combine
# ============================================================

final_plot <- region_strip / p +

  plot_layout(
    heights = c(
      0.08,
      1
    )
  )


final_plot

In [ ]:
# ============================================================
# 13. Save
# ============================================================

ggsave(
  file.path(
    OUT_DIR,
    "selected_cross_celltype_modules_LDSC_bubbleplot.pdf"
  ),
  final_plot,
  width = 7,
  height = 4.5,
  limitsize = FALSE
)


# ggsave(
#   file.path(
#     OUT_DIR,
#     "selected_cross_celltype_modules_LDSC_bubbleplot.png"
#   ),
#   final_plot,
#   width = 10,
#   height = 6.5,
#   dpi = 300,
#   limitsize = FALSE
# )